# Challenge 9: Enrich traces after the fact

**Extension of framework step 2: stratify by the segments that matter.** Step 2 says to sample by geography, customer tier and risk class. The production traces you seeded
carry `env`, `segment`, `release` and `thread_id`, and nothing else: nobody knew at trace time which customer an order belonged to. That is the usual state of real traces.
Fields like these live in another system, or arrive later, so you add them **after the fact** with `client.update_run`.

You will merge customer geography, customer tier, risk class and deployment facts into runs that are already in LangSmith, without losing what they carry, then use the new fields to select and compare traces.

## 1. At the source, or after the fact?

| The value is... | Add it... | How |
|---|---|---|
| Known by the code that runs the agent (the release, the model id) | **At the source** | pass `metadata=` and `tags=` when the run is created |
| In another system (an order or customer record), or computed from the result | **After the fact** | look it up later and `client.update_run(...)` |
| Needed on traces from a system you can't change | **After the fact** | same, keyed on something the trace does carry |

Run the setup cell. It reads one production run and shows what it carries today.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from collections import Counter

import enrich

roots = lab_sdk.roots(client)
print(len(roots), "root runs in", PROD)
sample = next(r for r in roots if "A1" in str(r.inputs))
print("\none run carries these metadata keys:", sorted(enrich.meta_of(sample)))
print("and these tags:", sample.tags)

## 2. The systems of record

The enrichment looks values up in two small registries that stand in for your real systems: an order system (who is the customer behind order `A1001`?) and a deployment record
(what did release `v1.1` ship?). Risk class is computed from what the run did: a run that called `issue_refund` moved money.

In [ ]:
import json

print("order registry:", json.dumps(enrich.ORDER_REGISTRY, indent=1)[:400], "...")
print("deployments   :", json.dumps(enrich.DEPLOYMENTS))
metadata, tags = enrich.registry_metadata(sample)
print("\nfor the run above, the enrichment would add:")
print("  metadata:", json.dumps(metadata))
print("  tags    :", tags)

## 3. Your turn: merge, don't overwrite

`update_run` takes `extra` and `tags`. The documentation doesn't say whether the `metadata` you send is **merged** into what a run already has or **replaces** it,
and a replace would wipe `release` and `segment`. So don't rely on it: read the run, merge locally, write the merged result.

Write `merge_into(run, metadata, tags)`: copy `run.extra`, merge `metadata` into its `metadata`, take the union of the existing tags and the new ones, and call `client.update_run` with the run id,
`extra`, `tags`, `end_time=run.end_time`, `trace_id=run.trace_id` and `dotted_order=run.dotted_order`.
The last two matter: these runs were already closed when they were created, and without them LangSmith rejects the update as a duplicate (409 Conflict).
A reference solution is filled in below. Try writing it yourself first, then compare.

In [ ]:
# Reference solution: try writing this yourself first.
def merge_into(run, metadata, tags):
    extra = dict(run.extra or {})
    extra["metadata"] = {**(extra.get("metadata") or {}), **metadata}
    client.update_run(
        run.id,
        extra=extra,
        tags=sorted(set(run.tags or []) | set(tags)),
        end_time=run.end_time,
        trace_id=run.trace_id,
        dotted_order=run.dotted_order,
    )

## 4. Enrich every root run

`enrich.apply` walks the root runs, skips any already stamped with `enrichment_version`, looks up the values, and calls your `merge_into`. Updates are applied asynchronously, so the next cell waits until they show up.

In [ ]:
assert callable(merge_into), "merge_into() must be defined in the blank cell above"
updated, already = enrich.apply(client, PROD, merge_into)
print(f"updated {updated} runs; {already} were already enriched")

done = enrich.wait_until_enriched(client, PROD, expected=updated + already)
print(f"{len(done)} root runs now carry enrichment_version={enrich.ENRICH_VERSION}")

from lab_state import mark_done
mark_done("/tmp/.sdk-ch9-merge-ok")

## 5. Nothing was lost

The point of read-merge-write: the old keys are still there next to the new ones.

In [ ]:
one = next(r for r in done if enrich.meta_of(r).get("customer_country") not in (None, "unknown"))
meta = enrich.meta_of(one)
print("kept   :", {k: meta.get(k) for k in ("release", "segment", "env")})
print("added  :", {k: meta.get(k) for k in ("customer_country", "customer_region", "customer_tier", "risk_class", "deploy_prompt_version")})
print("tags   :", one.tags)
assert meta.get("release") and meta.get("segment"), "release or segment is missing: merge_into replaced the metadata instead of merging it" 

## 6. Use the new fields

A new field is only worth having if you can query by it. Select by the metadata you just added, the same way challenge 1 selected by `release`, and compare a quality signal across a segment that did not exist before.

In [ ]:
since = lab_sdk.start_of_today()
gold = list(client.list_runs(
    project_name=PROD, is_root=True, start_time=since,
    filter='and(eq(metadata_key, "customer_tier"), eq(metadata_value, "gold"))'))
sea = list(client.list_runs(project_name=PROD, is_root=True, start_time=since, filter='has(tags, "region:sea")'))
print(f"gold-tier customers: {len(gold)} runs; region sea: {len(sea)} runs")

rows = {}
for r in client.list_runs(project_name=PROD, is_root=True, start_time=since):
    country = enrich.meta_of(r).get("customer_country", "n/a")
    row = rows.setdefault(country, {"runs": 0, "errors": 0, "process_ok": 0})
    row["runs"] += 1
    row["errors"] += int(bool(r.error))
    row["process_ok"] += lab_sdk.auto_score(r)

print(f"\n{'country':10}{'runs':>6}{'errors':>8}{'process score':>15}")
for country, v in sorted(rows.items()):
    print(f"{country:10}{v['runs']:>6}{v['errors']:>8}{v['process_ok'] / v['runs']:>15.2f}")

This is what the model upgrade framework needs in step 2 and step 5: a candidate has to be judged on each **segment**, and geography or customer tier are segments you could only add afterwards.
The same fields drive dashboards: group a chart by `customer_country` to see traffic by geography.

## 7. See it in LangSmith

In [ ]:
lab_links.project_link(PROD, hint="Filter: Metadata > customer_tier = gold, or Tags > region:sea. Monitor tab: group a chart by metadata customer_country")

## Optional: run it as a job

Because every run is stamped with `enrichment_version`, enrichment is **idempotent**: you can run it on a schedule against new traffic, and bump the version when your registry changes
to re-enrich old runs. This cell runs it a second time. (Optional: not checked.)

In [ ]:
# Optional cells run unless you set LAB_SKIP_OPTIONAL=1 before starting Jupyter. Nothing later depends on it.
OPTIONAL = os.environ.get("LAB_SKIP_OPTIONAL") != "1"
print("optional cells:", "on" if OPTIONAL else "off")

if OPTIONAL:
    updated, already = enrich.apply(client, PROD, merge_into)
    print(f"second run: updated {updated}, skipped {already} that were already enriched")